# **Tahap 06 - Classification and Prediction Modelling**

In [17]:
import pickle
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model  import LogisticRegression, Ridge, LinearRegression
from sklearn.ensemble      import RandomForestClassifier
from sklearn.metrics       import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score,
)
from sklearn.preprocessing import label_binarize
from xgboost               import XGBClassifier

warnings.filterwarnings('ignore')

## Path Konfigurasi

In [ ]:
BASE_DIR = (
    Path(__file__).resolve().parent.parent
    if "__file__" in dir()
    else Path.cwd().parent  
)

FEATURES_DIR = BASE_DIR / "data" / "features"
MODELS_DIR   = BASE_DIR / "models"

if not FEATURES_DIR.exists():
    raise FileNotFoundError(
        f"Folder data/features tidak ditemukan.\n"
        f"Jalankan dulu: python src/05_feature_engineering.py"
    )

MODELS_DIR.mkdir(parents=True, exist_ok=True)

## Konstanta

In [19]:
DANGER_LABELS = ['Rendah', 'Sedang', 'Tinggi', 'Berbahaya']

CLF_FEATURES = [
    'crime_rate_lag1',
    'crime_rate_lag2',
    'crime_rate_ma3',
    'crime_rate_yoy_change',
    'trend_direction_enc',
    'population_density',
    'unemployment_rate',
    'pulau_enc',
    'region_enc',
    'unemp_x_density',
    'crime_per_density',
]

CLF_TARGET = 'danger_level_enc'

REG_FEATURES = [
    'crime_rate_lag1',
    'crime_rate_ma3',
    'crime_rate_yoy_change',
    'trend_direction_enc',
    'population_density',
    'unemployment_rate',
    'pulau_enc',
    'unemp_x_density',
]

REG_TARGET = 'crime_rate_per100k'

## Helper Functions

In [20]:
def prepare_xy(df: pd.DataFrame, features: list, target: str) -> tuple:
    cols  = features + [target]
    clean = df[cols].dropna()
    X     = clean[features].values
    y     = clean[target].values
    idx   = clean.index
    return X, y, idx


def save_model(model, path: Path) -> None:
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    print(f"  Disimpan: {path.name}")


def print_section(title: str) -> None:
    print(f"\n{'=' * 60}")
    print(title)
    print('=' * 60)


def evaluate_classifier(name: str, model, X_test: np.ndarray, y_test: np.ndarray) -> dict:
    y_pred = model.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, average='macro', zero_division=0)
    cm     = confusion_matrix(y_test, y_pred)

    try:
        y_prob = model.predict_proba(X_test)
        y_bin  = label_binarize(y_test, classes=[0, 1, 2, 3])
        auc    = roc_auc_score(y_bin, y_prob, average='macro', multi_class='ovr')
    except Exception:
        auc = None

    report = classification_report(
        y_test, y_pred,
        target_names=DANGER_LABELS,
        output_dict=True,
        zero_division=0,
    )

    print(f"\n  {'-'*50}")
    print(f"  {name}")
    print(f"  {'-'*50}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1-macro : {f1:.4f}")
    if auc:
        print(f"  ROC-AUC  : {auc:.4f}")

    print(f"\n  Confusion Matrix:")
    print(f"  {'':12}", end='')
    for lbl in DANGER_LABELS:
        print(f"  {lbl[:8]:>8}", end='')
    print()
    for i, lbl in enumerate(DANGER_LABELS):
        print(f"  {lbl[:12]:12}", end='')
        for j in range(len(DANGER_LABELS)):
            val = cm[i][j] if i < len(cm) and j < len(cm[i]) else 0
            print(f"  {val:>8}", end='')
        print()

    print(f"\n  Per-class F1:")
    for lbl in DANGER_LABELS:
        if lbl in report:
            f1_cls = report[lbl]['f1-score']
            sup    = int(report[lbl]['support'])
            bar    = '#' * int(f1_cls * 15)
            print(f"    {lbl:<12} f1={f1_cls:.3f}  n={sup:>2}  {bar}")

    return {
        'model'   : name,
        'accuracy': round(acc, 4),
        'f1_macro': round(f1, 4),
        'roc_auc' : round(auc, 4) if auc else None,
        **{f'f1_{lbl.lower()}': round(report.get(lbl, {}).get('f1-score', 0), 4)
           for lbl in DANGER_LABELS},
    }


def get_feature_importance(model, feature_names: list, model_name: str) -> pd.DataFrame:
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importance = np.abs(model.coef_).mean(axis=0)
    else:
        return pd.DataFrame()

    df_imp = pd.DataFrame({
        'feature'   : feature_names,
        'importance': importance,
        'model'     : model_name,
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    df_imp['rank'] = range(1, len(df_imp) + 1)
    return df_imp


def evaluate_regressor(name: str, model, X_test: np.ndarray, y_test: np.ndarray) -> dict:
    y_pred = model.predict(X_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mae    = mean_absolute_error(y_test, y_pred)
    r2     = r2_score(y_test, y_pred)

    print(f"\n  {'-'*40}")
    print(f"  {name}")
    print(f"  {'-'*40}")
    print(f"  RMSE : {rmse:.2f}")
    print(f"  MAE  : {mae:.2f}")
    print(f"  R2   : {r2:.4f}")

    return {
        'model': name,
        'rmse' : round(rmse, 2),
        'mae'  : round(mae, 2),
        'r2'   : round(r2, 4),
    }

## Load Data

In [21]:
train = pd.read_csv(FEATURES_DIR / "features_train.csv")
test  = pd.read_csv(FEATURES_DIR / "features_test.csv")
train['tahun'] = train['tahun'].astype(int)
test['tahun']  = test['tahun'].astype(int)

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")

Train : (229, 28)
Test  : (102, 28)


## Bagian A - Klasifikasi

### Prepare Features

In [22]:
X_train, y_train, _        = prepare_xy(train, CLF_FEATURES, CLF_TARGET)
X_test,  y_test,  test_idx = prepare_xy(test,  CLF_FEATURES, CLF_TARGET)

print(f"Training set : {len(X_train)} samples (setelah drop NaN)")
print(f"Test set     : {len(X_test)} samples")
print(f"Fitur        : {len(CLF_FEATURES)}")
print(f"\nDistribusi y_train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Distribusi y_test : {dict(zip(*np.unique(y_test,  return_counts=True)))}")

Training set : 160 samples (setelah drop NaN)
Test set     : 102 samples
Fitur        : 11

Distribusi y_train: {0: 41, 1: 81, 2: 36, 3: 2}
Distribusi y_test : {0: 10, 1: 44, 2: 38, 3: 10}


### Logistic Regression (Baseline)

In [23]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    C=1.0,
)
lr.fit(X_train, y_train)
save_model(lr, MODELS_DIR / "clf_logistic.pkl")

  Disimpan: clf_logistic.pkl


### Random Forest

In [24]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
save_model(rf, MODELS_DIR / "clf_random_forest.pkl")

  Disimpan: clf_random_forest.pkl


### XGBoost

In [25]:
class_counts  = np.bincount(y_train.astype(int))
sample_weight = np.array([
    len(y_train) / (len(class_counts) * class_counts[int(yi)])
    for yi in y_train
])

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0,
)
xgb.fit(X_train, y_train, sample_weight=sample_weight)
save_model(xgb, MODELS_DIR / "clf_xgboost.pkl")

  Disimpan: clf_xgboost.pkl


### Evaluasi

In [26]:
clf_models = [
    ("Logistic Regression", lr),
    ("Random Forest",       rf),
    ("XGBoost",             xgb),
]

clf_results = []
for name, model in clf_models:
    res = evaluate_classifier(name, model, X_test, y_test)
    clf_results.append(res)

clf_report = pd.DataFrame(clf_results)
clf_report


  --------------------------------------------------
  Logistic Regression
  --------------------------------------------------
  Accuracy : 0.6667
  F1-macro : 0.5978
  ROC-AUC  : 0.7525

  Confusion Matrix:
                  Rendah    Sedang    Tinggi  Berbahay
  Rendah               6         2         2         0
  Sedang               5        32         6         1
  Tinggi               0         7        26         5
  Berbahaya            0         0         6         4

  Per-class F1:
    Rendah       f1=0.571  n=10  ########
    Sedang       f1=0.753  n=44  ###########
    Tinggi       f1=0.667  n=38  ##########
    Berbahaya    f1=0.400  n=10  ######

  --------------------------------------------------
  Random Forest
  --------------------------------------------------
  Accuracy : 0.5882
  F1-macro : 0.4877
  ROC-AUC  : 0.8959

  Confusion Matrix:
                  Rendah    Sedang    Tinggi  Berbahay
  Rendah               8         2         0         0
  Sedang     

,model,accuracy,f1_macro,roc_auc,f1_rendah,f1_sedang,f1_tinggi,f1_berbahaya
0,Logistic Regression,0.6667,0.5978,0.7525,0.5714,0.7529,0.6667,0.4000
1,Random Forest,0.5882,0.4877,0.8959,0.5333,0.6598,0.5758,0.1818
2,XGBoost,0.6078,0.5358,0.8318,0.5517,0.6735,0.5846,0.3333


### Feature Importance

In [27]:
imp_frames = []
for name, model in [("Random Forest", rf), ("XGBoost", xgb)]:
    imp = get_feature_importance(model, CLF_FEATURES, name)
    if not imp.empty:
        imp_frames.append(imp)

fi_df = pd.concat(imp_frames, ignore_index=True) if imp_frames else pd.DataFrame()
fi_df.head(20)

,feature,importance,model,rank
0,crime_rate_ma3,0.345596,Random Forest,1
1,crime_rate_lag1,0.180406,Random Forest,2
2,crime_rate_lag2,0.148634,Random Forest,3
3,crime_per_density,0.081920,Random Forest,4
4,crime_rate_yoy_change,0.076469,Random Forest,5
5,population_density,0.039971,Random Forest,6
6,region_enc,0.039111,Random Forest,7
7,unemp_x_density,0.037302,Random Forest,8
8,pulau_enc,0.020413,Random Forest,9
9,unemployment_rate,0.016198,Random Forest,10


### Prediction with the Best Model

In [28]:
best_preds = pd.DataFrame({
    'index'            : test_idx,
    'y_true_enc'       : y_test,
    'y_true_label'     : [DANGER_LABELS[int(y)] for y in y_test],
    'y_pred_enc_xgb'   : xgb.predict(X_test),
    'y_pred_label_xgb' : [DANGER_LABELS[int(p)] for p in xgb.predict(X_test)],
    'y_pred_enc_rf'    : rf.predict(X_test),
    'y_pred_label_rf'  : [DANGER_LABELS[int(p)] for p in rf.predict(X_test)],
})
best_preds.head(10)

,index,y_true_enc,y_true_label,y_pred_enc_xgb,y_pred_label_xgb,y_pred_enc_rf,y_pred_label_rf
0,0,1,Sedang,1,Sedang,1,Sedang
1,1,2,Tinggi,1,Sedang,1,Sedang
2,2,2,Tinggi,2,Tinggi,2,Tinggi
3,3,1,Sedang,0,Rendah,0,Rendah
4,4,2,Tinggi,1,Sedang,1,Sedang
5,5,2,Tinggi,2,Tinggi,2,Tinggi
6,6,0,Rendah,0,Rendah,0,Rendah
7,7,0,Rendah,0,Rendah,0,Rendah
8,8,0,Rendah,0,Rendah,0,Rendah
9,9,1,Sedang,1,Sedang,1,Sedang


## Bagian B - Regresi

### Prepare Features

In [29]:
X_train_r, y_train_r, _ = prepare_xy(train, REG_FEATURES, REG_TARGET)
X_test_r,  y_test_r,  _ = prepare_xy(test,  REG_FEATURES, REG_TARGET)

print(f"Training set : {len(X_train_r)} samples")
print(f"Test set     : {len(X_test_r)} samples")

Training set : 160 samples
Test set     : 102 samples


### Linear Regression

In [30]:
reg_lr = LinearRegression()
reg_lr.fit(X_train_r, y_train_r)
save_model(reg_lr, MODELS_DIR / "reg_linear.pkl")

  Disimpan: reg_linear.pkl


### Ridge Regression

In [31]:
ridge = Ridge(alpha=10.0, random_state=42)
ridge.fit(X_train_r, y_train_r)
save_model(ridge, MODELS_DIR / "reg_ridge.pkl")

  Disimpan: reg_ridge.pkl


### Results

In [32]:
reg_results = []
for name, model in [("Linear Regression", reg_lr), ("Ridge Regression", ridge)]:
    res = evaluate_regressor(name, model, X_test_r, y_test_r)
    reg_results.append(res)

reg_report = pd.DataFrame(reg_results)
reg_report


  ----------------------------------------
  Linear Regression
  ----------------------------------------
  RMSE : 72.12
  MAE  : 38.58
  R2   : 0.7607

  ----------------------------------------
  Ridge Regression
  ----------------------------------------
  RMSE : 72.03
  MAE  : 39.25
  R2   : 0.7613


,model,rmse,mae,r2
0,Linear Regression,72.12,38.58,0.7607
1,Ridge Regression,72.03,39.25,0.7613


## Simpan Laporan

In [33]:
clf_report.to_csv(FEATURES_DIR / "classification_report.csv", index=False)
print("  classification_report.csv")

if not fi_df.empty:
    fi_df.to_csv(FEATURES_DIR / "feature_importance.csv", index=False)
    print("  feature_importance.csv")

reg_report.to_csv(FEATURES_DIR / "regression_report.csv", index=False)
print("  regression_report.csv")

best_preds.to_csv(FEATURES_DIR / "predictions.csv", index=False)
print("  predictions.csv")

  classification_report.csv
  feature_importance.csv
  regression_report.csv
  predictions.csv


## Summary Perbandingan Model

In [34]:
print("Klasifikasi:")
print(clf_report[['model', 'accuracy', 'f1_macro', 'roc_auc']].to_string(index=False))
best_clf = clf_report.loc[clf_report['f1_macro'].idxmax(), 'model']
print(f"\nBest classifier : {best_clf}")

print("\nRegresi:")
print(reg_report.to_string(index=False))
best_reg = reg_report.loc[reg_report['r2'].idxmax(), 'model']
print(f"\nBest regressor  : {best_reg}")

print("\nTahap 6 selesai. Lanjut ke: dashboard/app.py")

Klasifikasi:
              model  accuracy  f1_macro  roc_auc
Logistic Regression    0.6667    0.5978   0.7525
      Random Forest    0.5882    0.4877   0.8959
            XGBoost    0.6078    0.5358   0.8318

Best classifier : Logistic Regression

Regresi:
            model  rmse   mae     r2
Linear Regression 72.12 38.58 0.7607
 Ridge Regression 72.03 39.25 0.7613

Best regressor  : Ridge Regression

Tahap 6 selesai. Lanjut ke: dashboard/app.py
